# 🚀 Deep Dive 02: Hybrid Search (RRF) & Cross-Encoder Reranking
### Part 2 of the DocuMind Learning Series

In **Notebook 01**, you learned why **BM25 Keyword Search** is essential for exact identifiers (`POL-8821`, `$4,821,000`, error codes) and why it fails on conceptual paraphrasing.

In this notebook, we build the full **Industry-Grade Hybrid Search & Reranking Pipeline**:

1. **Dense Vector Search**: Semantic embeddings (`BAAI/bge-small-en-v1.5`).
2. **Sparse Vector Search**: Token frequency weights (`Qdrant/bm25`).
3. **Reciprocal Rank Fusion (RRF)**: Merging both scoreboards into a unified ranking.
4. **Qdrant Native Hybrid Search**: Running dual prefetch and server-side RRF in a single query.
5. **Cross-Encoder Reranking**: Using deep self-attention to filter top candidates down to the highest-precision chunks.
6. **Side-by-Side Comparison**: Comparing all 4 retrieval strategies across edge-case queries.

In [1]:
# 1. Setup & Imports
import os
import uuid
import pandas as pd
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding
from sentence_transformers import CrossEncoder

# Configure pandas for clean tabular inspection
pd.set_option('display.max_colwidth', None)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Environment & models ready to initialize!")

d:\Abhi\Personal Projects\documind\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment & models ready to initialize!


---
## 2. Connect to Qdrant & Initialize Hybrid Collection

In Qdrant, a collection can hold **multiple named vectors** per point:
- `"dense"`: Standard dense semantic vector (384 dimensions) with Cosine distance.
- `"sparse"`: Sparse BM25 vector with inverted index.

*Note: We connect to Docker Qdrant (`http://localhost:6333`), and automatically fall back to fast in-memory Qdrant (`:memory:`) if Docker is offline.*

In [2]:
# Connect to Docker Qdrant or fallback to in-memory mode
try:
    client = QdrantClient(url="http://localhost:6333", timeout=2)
    client.get_collections()
    print("Connected to Docker Qdrant at http://localhost:6333")
except Exception:
    client = QdrantClient(location=":memory:")
    print("Running Qdrant in embedded in-memory mode (:memory:).")

COLLECTION_NAME = "documind_learning_hybrid"
DENSE_DIM = 384  # Dimension of BAAI/bge-small-en-v1.5

# Recreate collection for fresh experiment
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(
            size=DENSE_DIM,
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            index=models.SparseIndexParams(on_disk=False)
        )
    }
)

# Create a multi-tenant payload index for fast tenant filtering
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="tenant_id",
    field_schema=models.PayloadSchemaType.KEYWORD
)

print(f"Collection '{COLLECTION_NAME}' created with Dual Vectors (dense + sparse) & tenant_id index!")

Running Qdrant in embedded in-memory mode (:memory:).
Collection 'documind_learning_hybrid' created with Dual Vectors (dense + sparse) & tenant_id index!


C:\Users\slikh\AppData\Local\Temp\ipykernel_46052\227630134.py:33: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


d:\Abhi\Personal Projects\documind\venv\Lib\site-packages\qdrant_client\qdrant_remote.py:292: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


---
## 3. Embedding Models: Dense & Sparse

We initialize:
1. **Dense Model** (`BAAI/bge-small-en-v1.5`): Generates 384-dimensional semantic vectors.
2. **Sparse Model** (`Qdrant/bm25`): Generates token frequency coordinates (`indices` and `values`).

In [ ]:
print("Loading Dense Embedding model...")
dense_embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("Loading Sparse BM25 model...")
sparse_embedder = SparseTextEmbedding(model_name="Qdrant/bm25")

print("Both embedding models loaded successfully!")

---
## 4. Ingesting Documents with Dual Vectors

Let's index 6 realistic enterprise chunks. Each point in Qdrant receives:
- `id`: Unique integer or UUID
- `vector`: `{"dense": [...], "sparse": {"indices": [...], "values": [...]}}`
- `payload`: `{"tenant_id": "tenant_alpha", "text": "...", "chunk_id": ...}`

In [ ]:
documents = [
    "Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.",
    "Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%.",
    "Network Diagnostics Log: Production gateway node reported fatal error ERR_SSL_PROTOCOL_ERROR_443 on ingress port 443 during high concurrency traffic spike.",
    "HR Executive Directory: All medical leave accommodations must be authorized by Dr. Alexander Sterling in Employee Health & Wellbeing Services.",
    "Travel & Incidentals Policy: Submission guidelines for employee per-diem allowances, hotel accommodations, and mileage disbursements.",
    "Security & Firewall Protocols: Guide on managing TLS/SSL certificates, port forwarding on ports 80 and 443, and updating firewall ingress rules."
]

# Generate Dense and Sparse vectors for all chunks
dense_vectors = list(dense_embedder.embed(documents))
sparse_vectors = list(sparse_embedder.embed(documents))

points = []
for i, doc in enumerate(documents):
    point = models.PointStruct(
        id=i,
        vector={
            "dense": dense_vectors[i].tolist(),
            "sparse": models.SparseVector(
                indices=sparse_vectors[i].indices.tolist(),
                values=sparse_vectors[i].values.tolist()
            )
        },
        payload={
            "tenant_id": "tenant_alpha",
            "chunk_id": i,
            "text": doc
        }
    )
    points.append(point)

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Indexed {len(points)} dual-vector points into Qdrant collection '{COLLECTION_NAME}'!")

---
## 5. The Reciprocal Rank Fusion (RRF) Algorithm: Pure Python Walkthrough

Before calling Qdrant's internal C++ implementation, let's look at the **exact Python math** of RRF.

Given dense rankings and sparse rankings:
$$\text{RRF Score}(d) = \sum_{m \in \{\text{Dense}, \text{Sparse}\}} \frac{1}{k + \text{rank}_m(d)}$$

Where:
- $k = 60$ (smoothing constant).
- $\text{rank}_m(d)$ is the 1-based index of item $d$ in retriever $m$ ($1, 2, 3...$).

In [ ]:
def python_rrf(dense_ranks: list[int], sparse_ranks: list[int], k: int = 60) -> list[tuple[int, float]]:
    """
    Computes Reciprocal Rank Fusion scores for a set of document IDs.
    dense_ranks: ordered list of chunk IDs from dense search
    sparse_ranks: ordered list of chunk IDs from sparse search
    """
    rrf_scores = {}
    
    # Accumulate Dense ranks
    for rank, doc_id in enumerate(dense_ranks, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))
        
    # Accumulate Sparse ranks
    for rank, doc_id in enumerate(sparse_ranks, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))
        
    # Sort by RRF score descending
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs

# Toy test:
# Dense ranked: [0, 1, 2]
# Sparse ranked: [2, 0, 3]
print("Sample RRF Fusion calculation:")
for doc_id, score in python_rrf([0, 1, 2], [2, 0, 3]):
    print(f"Chunk {doc_id} -> RRF Score: {score:.5f}")

---
## 6. Native Qdrant Hybrid Search with Server-Side RRF

Qdrant executes hybrid search using `client.query_points()` with **`prefetch`** queries:
1. `prefetch` #1: Search `"dense"` vector space, return top candidates.
2. `prefetch` #2: Search `"sparse"` vector space, return top candidates.
3. `query = models.FusionQuery(fusion=models.Fusion.RRF)`: Qdrant merges both candidate lists inside the database engine and returns the top fused results in **one single call**!

In [ ]:
def search_qdrant_hybrid(query_text: str, tenant_id: str = "tenant_alpha", limit: int = 4):
    # 1. Embed query in dense and sparse spaces
    q_dense = list(dense_embedder.embed([query_text]))[0].tolist()
    q_sparse_raw = list(sparse_embedder.embed([query_text]))[0]
    q_sparse = models.SparseVector(
        indices=q_sparse_raw.indices.tolist(),
        values=q_sparse_raw.values.tolist()
    )
    
    # 2. Strict Multi-Tenant Filter
    tenant_filter = models.Filter(
        must=[models.FieldCondition(key="tenant_id", match=models.MatchValue(value=tenant_id))]
    )
    
    # 3. Native Qdrant Hybrid RRF Query
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=q_dense,
                using="dense",
                filter=tenant_filter,
                limit=limit * 2
            ),
            models.Prefetch(
                query=q_sparse,
                using="sparse",
                filter=tenant_filter,
                limit=limit * 2
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit
    )
    
    results = []
    for p in response.points:
        results.append({
            "Chunk ID": f"Chunk {p.payload['chunk_id']}",
            "RRF Score": round(p.score, 5),
            "Text": p.payload["text"]
        })
    return pd.DataFrame(results)

---
## 7. Cross-Encoder Reranking: Deep Attention Rescoring

Now we initialize the **Cross-Encoder** (`cross-encoder/ms-marco-MiniLM-L-6-v2`).

Unlike embeddings where query and document vectors are compared by dot product, a cross-encoder passes `[Query, Document]` through all 12 layers of multi-head self-attention together, predicting a precise relevance score.

In [ ]:
print("Loading Cross-Encoder Reranker model (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_chunks(query: str, candidate_texts: list[str]) -> list[tuple[int, float]]:
    # Create query-chunk pairs: [[query, text1], [query, text2], ...]
    pairs = [[query, text] for text in candidate_texts]
    scores = reranker.predict(pairs)
    
    # Return sorted indices and scores
    indexed_scores = [(idx, float(score)) for idx, score in enumerate(scores)]
    indexed_scores.sort(key=lambda x: x[1], reverse=True)
    return indexed_scores

print("Reranker initialized and ready!")

---
## 8. The Master Comparison Function: Dense vs. BM25 vs. Hybrid vs. Reranked

Let's build a unified function that runs all 4 retrieval strategies side-by-side on any query and displays the results in an inspection table!

In [ ]:
def compare_retrieval_strategies(query: str, top_k: int = 3):
    # 1. Dense Only
    q_dense = list(dense_embedder.embed([query]))[0].tolist()
    dense_res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_dense,
        using="dense",
        limit=top_k
    ).points
    dense_order = [f"Chunk {p.payload['chunk_id']} ({p.score:.3f})" for p in dense_res]
    
    # 2. Sparse Only
    q_sparse_raw = list(sparse_embedder.embed([query]))[0]
    q_sparse = models.SparseVector(indices=q_sparse_raw.indices.tolist(), values=q_sparse_raw.values.tolist())
    sparse_res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_sparse,
        using="sparse",
        limit=top_k
    ).points
    sparse_order = [f"Chunk {p.payload['chunk_id']} ({p.score:.3f})" for p in sparse_res]
    
    # 3. Hybrid RRF
    hybrid_df = search_qdrant_hybrid(query, limit=top_k * 2)
    hybrid_candidates = hybrid_df["Text"].tolist()
    hybrid_order = [f"{row['Chunk ID']} ({row['RRF Score']:.4f})" for _, row in hybrid_df.head(top_k).iterrows()]
    
    # 4. Hybrid + Reranker (Rerank top hybrid candidates)
    rerank_results = rerank_chunks(query, hybrid_candidates)
    rerank_order = []
    for original_idx, score in rerank_results[:top_k]:
        chunk_name = hybrid_df.iloc[original_idx]["Chunk ID"]
        rerank_order.append(f"{chunk_name} ({score:.3f})")
        
    # Pad lists if fewer results returned
    max_len = max(len(dense_order), len(sparse_order), len(hybrid_order), len(rerank_order))
    dense_order += ["-"] * (max_len - len(dense_order))
    sparse_order += ["-"] * (max_len - len(sparse_order))
    hybrid_order += ["-"] * (max_len - len(hybrid_order))
    rerank_order += ["-"] * (max_len - len(rerank_order))
    
    df = pd.DataFrame({
        "Rank": [f"#{i+1}" for i in range(max_len)],
        "1. Dense Only": dense_order,
        "2. Sparse Only": sparse_order,
        "3. Hybrid (RRF)": hybrid_order,
        "4. Hybrid + Reranker": rerank_order
    })
    
    print(f"\n{'='*70}")
    print(f"QUERY: '{query}'")
    print(f"{'='*70}")
    return df

---
## 9. Live Experiments: 3 Edge Cases That Reveal Everything

### Test Case 1: Exact Technical Error Code (`ERR_SSL_PROTOCOL_ERROR_443`)
- Target Answer: **Chunk 2**

In [ ]:
compare_retrieval_strategies("How do I fix ERR_SSL_PROTOCOL_ERROR_443 on port 443?")

### Test Case 2: Conceptual Paraphrase (No Shared Vocabulary)
- Target Answer: **Chunk 4** (`"employee per-diem allowances, hotel accommodations, and mileage disbursements"`)
- Query uses: *"money back"*, *"work trip"*, *"reimbursement"*

In [ ]:
compare_retrieval_strategies("How do I get my money back for a work trip reimbursement?")

### Test Case 3: Mixed Query (Financial Number + Concept)
- Query combines exact numbers (`"$4,821,000"`) and concepts (`"growth rate"`, *"SaaS earnings"*)
- Target Answer: **Chunk 1**

In [ ]:
compare_retrieval_strategies("Which quarter had $4,821,000 in SaaS earnings and what was the growth rate?")

---
## Summary: What You Mastered in this Lab

1. **Dense Search alone** misses exact codes and specific error strings because semantic compression dilutes exact token IDs.
2. **BM25 Search alone** produces complete zero-score misses when the user paraphrases or uses natural language.
3. **Reciprocal Rank Fusion (RRF)** seamlessly merges both without score normalization bugs.
4. **Cross-Encoder Reranking** provides that final high-precision polish, weeding out "accidental" keyword matches and prioritizing the truly relevant chunk for the LLM.

You now understand the core retrieval engine powering **DocuMind**!